# Bonus: Hidden Incentivized Reviews

<div dir="rtl">

این نوت‌بوک یک مسئله‌ی جدید روی داده‌ی نظرهای آمازون است: پیدا کردن نظرهای تشویقی بعد از ممنوعیت آمازون و سنجیدن اثرشان روی امتیاز محصول. نظر تشویقی نظری است که نویسنده‌اش در ازای آن کالا را مجانی یا با تخفیف گرفته است.

تا ۳ اکتبر ۲۰۱۶ آمازون این کار را مجاز می‌دانست، به شرطی که نویسنده آن را در متن نظر اعلام کند. از آن روز نظرهای تشویقی ممنوع شدند و فقط برنامه‌ی Vine خود آمازون استثنا ماند ([TechCrunch](https://techcrunch.com/2016/10/03/amazon-bans-incentivized-reviews-tied-to-free-or-discounted-products/)). به نقل همین گزارش، ReviewMeta در بیش از ۷ میلیون نظر دیده بود که میانگین امتیاز محصولات دارای نظر تشویقی ۴.۷۴ است، در برابر ۴.۳۶ برای بقیه. FTC هم در اوت ۲۰۲۴ نظرهای جعلی و پاداشی را ممنوع کرد و برای متخلفان جریمه گذاشت ([FTC](https://www.ftc.gov/news-events/news/press-releases/2024/08/federal-trade-commission-announces-final-rule-banning-fake-reviews-testimonials)). پس این مسئله هنوز برای فروشگاه‌ها موضوع حقوقی و مالی است.

**ایده:** قبل از ممنوعیت نویسنده‌ها باید اعلام می‌کردند، پس نظرهای تشویقی آن دوره با یک regex و بدون برچسب‌زنی دستی پیدا می‌شوند. جمله‌ی اعلام از همه‌ی متن‌ها حذف می‌شود و مدل یاد می‌گیرد نظر تشویقی را از روی بقیه‌ی متن تشخیص دهد. بعد از ممنوعیت جمله‌ی اعلام تقریباً ناپدید می‌شود و مدل روی نظرهای این دوره اجرا می‌شود. درستی‌سنجی این دوره هم برچسب دستی لازم ندارد: یک بار با تاریخچه‌ی نظردهنده‌ها و یک بار با حلقه‌های نظردهی که مستقل از متن‌اند.

**خروجی بیزینسی:** امتیاز مشکوک بودن هر نظر برای صف بررسی، سهم نظرهای مشکوک هر محصول و برند، و امتیاز اصلاح‌شده‌ی هر محصول.

**مراحل:**
1. **Data:** بار کردن داده، حذف نظرهای تکراری و ستون تاریخ.
2. **The 2016 ban:** روند ماهانه‌ی نظرهای اعلام‌شده، تورم امتیاز داخل همان محصول‌ها و تغییر امتیاز محصول‌ها بعد از ممنوعیت.
3. **Weak labels، Training pool و Masking:** ساختن برچسب از جمله‌ی اعلام، داده‌ی آموزش قبل از ممنوعیت و حذف نشانه‌های اعلام از متن.
4. **Model comparison:** مقایسه‌ی مدل‌های کلاسیک روی TF-IDF و بردارهای MiniLM و یک MiniLM که fine-tune شده، با cross-validation.
5. **After the ban:** اجرای مدل روی نظرهای بعد از ممنوعیت و سنجیدنش با نظردهنده‌هایی که در آموزش نبودند.
6. **Review rings:** آزمون مستقل از متن با گروه‌هایی از نظردهنده‌ها که روی محصول‌های کم‌نظر مشترک نظر داده‌اند.
7. **Fine-tuned MiniLM after the ban:** مقایسه‌ی MiniLM fine-tune شده با مدل نهایی روی همان آزمون بعد از ممنوعیت.
8. **Business output:** صف بررسی، محصول‌ها و برندهای مشکوک و امتیاز اصلاح‌شده‌ی محصول‌ها.

فایل‌های لازم در پوشه‌ی project2 در Google Drive: train_data.csv و title_brand.csv. fine-tune کردن MiniLM به GPU نیاز دارد و در نوت‌بوک ضمیمه‌ی bonus_minilm_finetune.ipynb انجام شده است. این نوت‌بوک ورودی‌های آن (pool.csv.gz و post_sample.csv.gz) را می‌سازد و نتیجه‌هایش (minilm_finetune_cv.csv و post_sample_finetune.csv) را از پوشه‌ی project2/bonus می‌خواند. همه‌ی خروجی‌ها، از جمله فایل‌های داشبورد، در همان پوشه ذخیره می‌شوند.

</div>

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import os
import re
import html
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = 'plotly_white'
path = '/content/drive/MyDrive/project2/'
out = path + 'bonus/'
os.makedirs(out, exist_ok=True)

## Data

<div dir="rtl">

مثل بخش ۱، نظرهای تکراری (همان نویسنده، همان محصول، همان متن) و asinهای تکراری جدول محصولات حذف شدند. ستون verified برای درستی‌سنجی بعد از ممنوعیت و تاریخ برای جدا کردن دو دوره لازم است.

</div>

In [3]:
cols = ['overall', 'verified', 'reviewerID', 'asin', 'reviewText', 'summary', 'unixReviewTime']
train = pd.read_csv(path + 'train_data.csv', usecols=cols)
products = pd.read_csv(path + 'title_brand.csv')
print('reviews:', train.shape, ' products:', products.shape)

reviews: (838944, 7)  products: (786445, 3)


In [4]:
key = ['reviewerID', 'asin', 'reviewText']
print('duplicate reviews:', train.duplicated(key).sum())
train = train.drop_duplicates(key).reset_index(drop=True)

products = products.drop_duplicates('asin')
products['title'] = products['title'].map(html.unescape, na_action='ignore')
print('reviews after dropping duplicates:', len(train))
train.isna().sum()

duplicate reviews: 8554


reviews after dropping duplicates: 830390


overall            0
verified           0
reviewerID         0
asin               0
reviewText         0
summary           74
unixReviewTime     0
dtype: int64

In [5]:
ban = pd.Timestamp('2016-10-03')
train['date'] = pd.to_datetime(train['unixReviewTime'], unit='s')
train['pre_ban'] = train['date'] < ban

print('first review:', train['date'].min().date(), ' last review:', train['date'].max().date())
print('reviews before the ban:', train['pre_ban'].sum(), f"({train['pre_ban'].mean():.1%})")
print('reviews after the ban:', (~train['pre_ban']).sum())

first review: 2016-01-01  last review: 2018-10-03
reviews before the ban: 370740 (44.6%)
reviews after the ban: 459650


<div dir="rtl">

داده از ۱ ژانویه‌ی ۲۰۱۶ تا ۳ اکتبر ۲۰۱۸ است. فقط ۹ ماه از آن قبل از ممنوعیت است، ولی چون تعداد نظرها در ماه‌های اول بیشتر است، ۴۴.۶٪ نظرها مال همین دوره‌اند.

</div>

## The 2016 ban

<div dir="rtl">

نظرهای تشویقی قبل از ممنوعیت با جمله‌ای مثل "I received this product at a discount in exchange for my honest review" اعلام می‌شدند. این regex دو شکل اصلی این جمله را می‌گیرد: اسم امتیاز (discount، free، reduced price یا sample) که حداکثر ۸۰ کاراکتر بعدش، در همان جمله، "in exchange for" یا "for my honest" آمده باشد، و خود عبارت "in exchange for my honest/unbiased". دقت این regex و تعداد نظرهایی که از دستش در می‌روند در بخش Weak labels بررسی می‌شود.

</div>

In [6]:
disclosure = (r'(?:discount|free|reduced price|sample)[^.]{0,80}(?:in exchange for|for (?:my|an) (?:honest|unbiased))'
              r'|in exchange for (?:my|an) (?:honest|unbiased)')
train['disclosed'] = train['reviewText'].str.lower().str.contains(disclosure)

print('disclosed reviews:', train['disclosed'].sum(), f"({train['disclosed'].mean():.2%})")
print('before the ban:', (train['disclosed'] & train['pre_ban']).sum(), ' after the ban:', (train['disclosed'] & ~train['pre_ban']).sum())
print('mean rating, disclosed:', round(train.loc[train['disclosed'], 'overall'].mean(), 2),
      ' others:', round(train.loc[~train['disclosed'], 'overall'].mean(), 2))

disclosed reviews: 11965 (1.44%)
before the ban: 11917  after the ban: 48
mean rating, disclosed: 4.62  others: 4.01


### Monthly trends

<div dir="rtl">

سهم نظرهای اعلام‌شده، میانگین امتیاز و سهم verified برای هر ماه حساب شد. میانگین امتیاز و سهم verified یک بار برای همه‌ی نظرها و یک بار بدون نظرهای اعلام‌شده آمده تا معلوم شود چه مقدار از تغییرها فقط به‌خاطر حذف همین نظرهاست. اکتبر ۲۰۱۸ فقط ۳ روز در داده دارد و کنار گذاشته شد.

</div>

In [7]:
train['month'] = train['date'].dt.to_period('M').dt.to_timestamp()
rest = train[~train['disclosed']]

monthly = pd.DataFrame({'reviews': train.groupby('month').size(),
                        'disclosed': train.groupby('month')['disclosed'].mean(),
                        'stars': train.groupby('month')['overall'].mean(),
                        'stars_rest': rest.groupby('month')['overall'].mean(),
                        'verified': train.groupby('month')['verified'].mean(),
                        'verified_rest': rest.groupby('month')['verified'].mean()})
monthly = monthly[monthly['reviews'] >= 1000]
monthly.loc['2016-01':'2017-01'].round(3)

,reviews,disclosed,stars,stars_rest,verified,verified_rest
month,,,,,,
2016-01-01,48259,0.024,4.070,4.058,0.834,0.853
2016-02-01,41165,0.025,4.073,4.059,0.839,0.861
2016-03-01,41742,0.028,4.067,4.052,0.835,0.859
2016-04-01,38153,0.035,4.075,4.057,0.824,0.853
2016-05-01,39348,0.033,4.081,4.061,0.821,0.847
2016-06-01,40328,0.072,4.096,4.055,0.782,0.839
2016-07-01,43005,0.055,4.090,4.059,0.788,0.829
2016-08-01,40469,0.012,4.062,4.055,0.830,0.839
2016-09-01,36260,0.004,4.069,4.066,0.845,0.848


In [8]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=['Share of disclosed incentivized reviews', 'Mean rating', 'Share of verified purchases'])
fig.add_trace(go.Scatter(x=monthly.index, y=monthly['disclosed'], name='disclosed share',
                         line_color='#2CA02C', customdata=monthly['reviews'],
                         hovertemplate='%{x|%b %Y}<br>%{y:.1%}<br>%{customdata:,} reviews<extra></extra>'), row=1, col=1)
for row, col in [(2, 'stars'), (3, 'verified')]:
    fig.add_trace(go.Scatter(x=monthly.index, y=monthly[col], name='all reviews', line_color='#1F77B4',
                             legendgroup='all', showlegend=row == 2), row=row, col=1)
    fig.add_trace(go.Scatter(x=monthly.index, y=monthly[col + '_rest'], name='without disclosed reviews',
                             line_color='#FF7F0E', legendgroup='rest', showlegend=row == 2), row=row, col=1)
fig.add_vline(x=ban, line_dash='dash', line_color='gray', row='all', col=1)
fig.add_vline(x=ban.timestamp() * 1000, line_width=0, annotation_text='ban: 2016-10-03', row=1, col=1)
fig.update_yaxes(tickformat='.0%', row=1, col=1)
fig.update_yaxes(tickformat='.0%', row=3, col=1)
fig.update_layout(height=750, hovermode='x unified', legend=dict(orientation='h', y=-0.08))
fig.show()

<div dir="rtl">

- سهم نظرهای اعلام‌شده در ژانویه تا مه ۲۰۱۶ بین ۲.۴٪ و ۳.۵٪ بود و در ژوئن به ۷.۲٪ رسید. از اوت پایین آمد (۱.۲٪ در اوت و ۰.۴٪ در سپتامبر) و بعد از ممنوعیت به صفر رسید.
- قبل از ممنوعیت میانگین همه‌ی نظرها بالاتر از میانگین بدون نظرهای اعلام‌شده بود، در ژوئن ۴.۰۹۶ در برابر ۴.۰۵۵. بعد از ممنوعیت دو خط یکی شدند. پس اثر مستقیم این نظرها روی میانگین کل فروشگاه کوچک است، ولی همین اثر روی چند هزار محصول جمع شده است.
- میانگین امتیاز در ۲۰۱۷ و ۲۰۱۸ در هر دو خط پایین‌تر رفت (حدود ۳.۹ در ۲۰۱۸). این روند در نظرهای بدون اعلام هم هست، پس نمی‌شود آن را به ممنوعیت نسبت داد.
- سهم verified از ۰.۷۸ در ژوئن ۲۰۱۶ به ۰.۸۹۵ در ژانویه‌ی ۲۰۱۷ رسید. بیشتر این جهش به‌خاطر حذف نظرهای اعلام‌شده است، ولی در نظرهای بدون اعلام هم سهم verified از بین ۰.۸۳ و ۰.۸۶ (ژانویه تا سپتامبر ۲۰۱۶) به ۰.۸۹۵ رسید.

</div>

In [9]:
print('verified share, disclosed reviews:', round(train.loc[train['disclosed'], 'verified'].mean(), 3))
print('verified share, other reviews before the ban:', round(train.loc[~train['disclosed'] & train['pre_ban'], 'verified'].mean(), 3))

verified share, disclosed reviews: 0.05
verified share, other reviews before the ban: 0.848


<div dir="rtl">

فقط ۵٪ نظرهای اعلام‌شده verified‌اند، چون کالا با قیمت معمول خریده نشده است. این تفاوت در بخش After the ban به‌عنوان یک شاخص کمکی به کار می‌آید.

</div>

### Rating inflation within the same products

<div dir="rtl">

مقایسه‌ی خام ۴.۶۲ با ۴.۰۱ می‌تواند فقط به این دلیل باشد که نظرهای تشویقی بیشتر برای محصول‌های خوب نوشته شده‌اند. برای جدا کردن اثر محصول، مثل بخش ۲، میانگین نظرهای اعلام‌شده‌ی هر محصول با میانگین بقیه‌ی نظرهای همان محصول مقایسه شد. فقط دوره‌ی قبل از ممنوعیت و محصول‌هایی آمده‌اند که دست‌کم ۵ نظر دیگر دارند.

</div>

In [10]:
pre = train[train['pre_ban']]
per = pre.groupby(['asin', 'disclosed'])['overall'].agg(['size', 'mean']).unstack()
per.columns = ['n_rest', 'n_disclosed', 'mean_rest', 'mean_disclosed']

both = per[(per['n_disclosed'] >= 1) & (per['n_rest'] >= 5)].copy()
both['gap'] = both['mean_disclosed'] - both['mean_rest']
print('products:', len(both))
print('mean gap:', round(both['gap'].mean(), 2), ' median gap:', round(both['gap'].median(), 2))
print('share of products with higher disclosed reviews:', round((both['gap'] > 0).mean(), 3))
print('share with lower disclosed reviews:', round((both['gap'] < 0).mean(), 3))

products: 2344
mean gap: 0.51  median gap: 0.5
share of products with higher disclosed reviews: 0.777
share with lower disclosed reviews: 0.177


In [11]:
from scipy.stats import wilcoxon

stat, p = wilcoxon(both['gap'])
print('Wilcoxon signed-rank p-value:', p)

Wilcoxon signed-rank p-value: 6.407544481765454e-179


In [12]:
fig = px.histogram(both, x='gap', nbins=40,
                   title='Disclosed reviews minus other reviews of the same product (before the ban)')
fig.add_vline(x=0, line_dash='dash')
fig.update_layout(xaxis_title='rating gap (stars)', yaxis_title='products')
fig.show()

In [13]:
pd.crosstab(pre['disclosed'], pre['overall'], normalize='index').round(3)

overall,1,2,3,4,5
disclosed,,,,,
False,0.091,0.065,0.095,0.192,0.556
True,0.010,0.012,0.044,0.221,0.713


<div dir="rtl">

در ۲٬۳۴۴ محصول، نظرهای اعلام‌شده به‌طور میانگین ۰.۵۱ ستاره بالاتر از بقیه‌ی نظرهای همان محصول‌اند و در ۷۸٪ محصول‌ها بالاترند. یعنی بیشتر فاصله‌ی خام ۰.۶۱ ستاره‌ای به خود تشویق برمی‌گردد، نه به انتخاب محصول‌های بهتر. آزمون Wilcoxon signed-rank به کار رفت چون برای هر محصول یک فاصله داریم (داده‌ی جفتی) و توزیع فاصله‌ها نرمال نیست. p-value آن عملاً صفر است. در توزیع ستاره‌ها هم ۷۱٪ نظرهای اعلام‌شده ۵ ستاره‌اند، در برابر ۵۶٪ بقیه، و فقط ۲٪ آن‌ها ۱ یا ۲ ستاره‌اند، در برابر ۱۶٪.

</div>

### Products after the ban

<div dir="rtl">

اگر نظرهای تشویقی امتیاز محصول را بالا برده باشند، امتیاز محصول‌هایی که این نظرها را داشتند بعد از ممنوعیت باید بیشتر از بقیه پایین بیاید. چون میانگین کل داده در این سال‌ها پایین می‌رود، تغییر این محصول‌ها با تغییر محصول‌های بدون نظر اعلام‌شده مقایسه شد (difference-in-differences). فقط محصول‌هایی آمده‌اند که هم قبل و هم بعد از ممنوعیت دست‌کم ۱۰ نظر دارند.

</div>

In [14]:
post = train[~train['pre_ban']]
prod = pre.groupby('asin').agg(n_pre=('overall', 'size'), mean_pre=('overall', 'mean'), n_disclosed=('disclosed', 'sum'))
prod['mean_pre_rest'] = pre[~pre['disclosed']].groupby('asin')['overall'].mean()
prod = prod.join(post.groupby('asin').agg(n_post=('overall', 'size'), mean_post=('overall', 'mean')), how='inner')
prod = prod[(prod['n_pre'] >= 10) & (prod['n_post'] >= 10)].copy()

prod['group'] = np.where(prod['n_disclosed'] > 0, 'had disclosed reviews', 'no disclosed reviews')
prod['change'] = prod['mean_post'] - prod['mean_pre']
prod['change_rest'] = prod['mean_post'] - prod['mean_pre_rest']

summary = prod.groupby('group').agg(products=('change', 'size'), mean_pre=('mean_pre', 'mean'),
                                    mean_pre_rest=('mean_pre_rest', 'mean'), mean_post=('mean_post', 'mean'),
                                    change=('change', 'mean'), change_rest=('change_rest', 'mean'))
summary.round(3)

,products,mean_pre,mean_pre_rest,mean_post,change,change_rest
group,,,,,,
had disclosed reviews,690,4.219,4.174,3.932,-0.287,-0.243
no disclosed reviews,4222,4.139,4.139,3.968,-0.171,-0.171


In [15]:
from scipy.stats import mannwhitneyu

treated = prod[prod['group'] == 'had disclosed reviews']
control = prod[prod['group'] == 'no disclosed reviews']
print('difference-in-differences:', round(treated['change'].mean() - control['change'].mean(), 3))
print('same, without disclosed reviews before the ban:', round(treated['change_rest'].mean() - control['change'].mean(), 3))
print('Mann-Whitney p-value:', mannwhitneyu(treated['change'], control['change']).pvalue)

difference-in-differences: -0.116
same, without disclosed reviews before the ban: -0.072
Mann-Whitney p-value: 2.285180352052974e-09


<div dir="rtl">

محصول‌هایی که قبل از ممنوعیت امتیاز بالاتری داشتند در هر حال جا برای افت بیشتری دارند (بازگشت به میانگین). برای همین مقایسه داخل گروه‌هایی با امتیاز مشابه قبل از ممنوعیت هم تکرار شد.

</div>

In [16]:
prod['level'] = pd.cut(prod['mean_pre'], [1, 3.5, 4, 4.3, 4.6, 5], include_lowest=True,
                       labels=['1-3.5', '3.5-4', '4-4.3', '4.3-4.6', '4.6-5'])
by_level = prod.groupby(['level', 'group'], observed=True)['change'].agg(['mean', 'size'])
display(by_level.unstack().round(3))

by_level = by_level.reset_index()

fig = px.bar(by_level, x='level', y='mean', color='group', barmode='group', text=by_level['mean'].round(2),
             hover_data=['size'], color_discrete_map={'had disclosed reviews': '#D62728', 'no disclosed reviews': '#7F7F7F'},
             title='Change in mean rating after the ban, by mean rating before the ban')
fig.update_layout(xaxis_title='mean rating before the ban', yaxis_title='mean after minus mean before')
fig.show()

mean                                       size  \
group   had disclosed reviews no disclosed reviews had disclosed reviews   
level                                                                      
1-3.5                   0.107                0.199                    44   
3.5-4                  -0.159               -0.107                   153   
4-4.3                  -0.261               -0.200                   175   
4.3-4.6                -0.354               -0.252                   193   
4.6-5                  -0.513               -0.345                   125   

                              
group   no disclosed reviews  
level                         
1-3.5                    417  
3.5-4                   1111  
4-4.3                    991  
4.3-4.6                 1079  
4.6-5                    624

<div dir="rtl">

محصول‌هایی که نظر اعلام‌شده داشتند بعد از ممنوعیت ۰.۲۹ ستاره افت کردند و بقیه ۰.۱۷، یعنی ۰.۱۲ ستاره افت اضافه (Mann-Whitney، p کمتر از ۰.۰۰۱). حدود ۰.۰۴ از آن به‌خاطر کنار رفتن خود نظرهای اعلام‌شده است. ۰.۰۷ باقی‌مانده یعنی نظرهای بدون اعلامِ این محصول‌ها هم قبل از ممنوعیت بالاتر بوده‌اند: یا بخشی از نظرهای تشویقی اعلام نشده بودند، یا این محصول‌ها با گذشت زمان ناراضی‌کننده‌تر شده‌اند. این داده بین این دو فرق نمی‌گذارد. داخل گروه‌های هم‌امتیاز هم این محصول‌ها در ۴ گروه از ۵ گروه بیشتر افت کرده‌اند (برای امتیاز بالای ۴.۶، ۰.۵۱ در برابر ۰.۳۴). استثنا گروه زیر ۳.۵ است که فقط ۴۴ محصول دارد.

</div>

## Weak labels

<div dir="rtl">

برچسب‌ها از خود متن می‌آیند و دستی زده نشده‌اند. نظری که قبل از ممنوعیت جمله‌ی اعلام دارد مثبت است. بقیه‌ی نظرها منفی قطعی نیستند، فقط برچسب ندارند، چون ممکن است تشویقی باشند و اعلام نکرده باشند. این ۱۰ نمونه‌ی تصادفی فقط برای دیدن کار regex است.

</div>

In [17]:
sample = train[train['disclosed']].sample(10, random_state=1)
for text in sample['reviewText']:
    m = re.search(disclosure, text.lower())
    print('...', text[max(m.start() - 100, 0): m.end() + 40].replace('\n', ' '), '...\n')

... ere.  I am giving it four stars because it arrived with a small hole.  I received this product at a discount for an honest and unbiased review.  My opinions are m ...

... jhS.png" class="video-slate-img-url">Im thrilled I was able to purchase this portable massager at a reduced price in exchange for an unbiased product evaluation and review.  This ma ...

... I am so glad that I was able to review this item at a discounted price in exchange for my honest and unbiased review:  I always have pro ...

... , I don't have to charge them up half way through the day.  Very cool!  I received this product for free or at a discount for an honest review. ...

... hoppers. All reviews I write reflect my honest opinion of the product. I received this product at a discount in exchange for an honest and unbiased review. ...

... n off/on switch. The charging stations us rated at the newest 3.0  Disclosure: I received this at a discounted price in exchange for product review and testing for my un

<div dir="rtl">

regex همه‌ی شکل‌های اعلام را نمی‌گیرد. چند عبارت دیگر که در اعلام‌ها زیاد دیده می‌شوند جدا جست‌وجو شدند.

</div>

In [18]:
other_forms = (r'honest review|unbiased review|disclaimer|in exchange for (?:a|my|an)? ?review|free of charge in exchange'
               r'|(?:at a|for a) discount (?:for|in exchange)|received (?:this|the|it) (?:product |item )?(?:for free|free|at a discount|at a reduced)')
train['maybe'] = train['reviewText'].str.lower().str.contains(other_forms) & ~train['disclosed']

for name, period in [('before the ban', train['pre_ban']), ('after the ban', ~train['pre_ban'])]:
    n = (train['maybe'] & period).sum()
    print(f'{name}: {n} reviews ({n / period.sum():.2%})')

before the ban: 4282 reviews (1.15%)
after the ban: 533 reviews (0.12%)


In [19]:
for text in train.loc[train['maybe'] & train['pre_ban'], 'reviewText'].sample(6, random_state=1):
    m = re.search(other_forms, text.lower())
    print('...', text[max(m.start() - 100, 0): m.end() + 60].replace('\n', ' '), '...\n')

... en this would be a good option. I received this tablet case complimentary in exchange of a fair and honest review, these opinions are my own. ...

... . It clings well to most surfaces. Surprising that a small speaker can project such a good sound. I received this product at a reduced price. ...

... s without having try and hook up to wifi. He said he would like to get another it worked so well. I received this at a discount to review. ...

... ra car port is a major bonus also. I am looking forard to using this a lot on my summer road trips. Disclaimer: I did contact the seller and request if I could receive a  ...

...  helps with grip.  Disclosure: I received this product for a discounted cost to test and provide an honest review. ...

... his TurnRaise Trucker Over the Head Bluetooth Wireless Headset w/ Noise Cancelling & Boom Mic.  *** Disclaimer I received this TurnRaise Trucker Over the Head Bluetooth W ...



<div dir="rtl">

۴٬۲۸۲ نظر قبل از ممنوعیت یکی از این عبارت‌ها را دارند و بعد از ممنوعیت سهمشان حدود ۱۰ برابر کم شده است (از ۱.۱۵٪ به ۰.۱۲٪). پس بیشترشان اعلام‌هایی بوده‌اند که regex نگرفته. ولی همه‌شان اعلام نیستند، مثلاً "honest review" یا "disclaimer" در نظرهای معمولی هم می‌آید. برای همین این نظرها نه مثبت حساب شدند و نه منفی، و از داده‌ی آموزش کنار می‌روند.

این وضعیت در یادگیری ماشین PU learning نام دارد: چند مثبت قطعی داریم و بقیه‌ی داده برچسب ندارد. Elkan و Noto (۲۰۰۸) نشان دادند که اگر مثبت‌های برچسب‌دار نمونه‌ی تصادفی از همه‌ی مثبت‌ها باشند، مدلی که مثبت را از بی‌برچسب جدا می‌کند نمونه‌ها را به همان ترتیبی مرتب می‌کند که مدل مثبت در برابر منفی واقعی. پس احتمال خروجی مدل دقیق نیست ولی ترتیبش درست است، و معیار ارزیابی باید رتبه‌بندی باشد (AUC و average precision)، نه accuracy. ساختن برچسب از یک نشانه در خود داده به‌جای برچسب‌زنی دستی هم distant supervision نام دارد، مثل کار Go و همکاران (۲۰۰۹) که شکلک‌های توییت را برچسب احساس گرفتند.

</div>

## Training pool

<div dir="rtl">

مدل فقط با نظرهای قبل از ممنوعیت آموزش می‌بیند، چون فقط آن دوره برچسب دارد. سه قاعده برای جلوگیری از نشت داده و اثرهای ناخواسته:
- ۳۰٪ نظردهنده‌هایی که دست‌کم یک نظر اعلام‌شده دارند، با همه‌ی نظرهایشان، از آموزش کنار گذاشته می‌شوند. نظرهای بعد از ممنوعیتِ این افراد آزمون بیرونی بخش After the ban است و مدل نباید هیچ نوشته‌ای از آن‌ها دیده باشد.
- فقط محصول‌هایی می‌آیند که دست‌کم یک نظر اعلام‌شده دارند، تا مثبت و منفی از همان محصول‌ها باشند و مدل نوع کالا را به جای سبک نوشتن یاد نگیرد.
- امتیاز ستاره‌ای و verified ورودی مدل نیستند. اولی مدل را به تشخیص‌دهنده‌ی نظر ۵ ستاره تبدیل می‌کند و دومی برای آزمون بعد از ممنوعیت لازم است.

</div>

In [20]:
pre = train[train['pre_ban']]
incentivized = pre.loc[pre['disclosed'], 'reviewerID'].drop_duplicates()
held_out = set(incentivized.sample(frac=0.3, random_state=0))
print('reviewers with a disclosed review:', len(incentivized), ' held out:', len(held_out))

keep = pre[~pre['reviewerID'].isin(held_out) & ~pre['maybe']]
pool = keep[keep['asin'].isin(keep.loc[keep['disclosed'], 'asin'])].reset_index(drop=True)
print('reviews:', len(pool), ' positives:', pool['disclosed'].sum(), f"({pool['disclosed'].mean():.1%})")
print('products:', pool['asin'].nunique(), ' reviewers:', pool['reviewerID'].nunique())

reviewers with a disclosed review: 4489  held out: 1347
reviews: 42321  positives: 8388 (19.8%)
products: 5552  reviewers: 29132


<div dir="rtl">

داده‌ی آموزش ۴۲٬۳۲۱ نظر روی ۵٬۵۵۲ محصول است و ۱۹.۸٪ آن مثبت است. ۱٬۳۴۷ نظردهنده برای آزمون بعد از ممنوعیت کنار رفتند.

</div>

## Masking

<div dir="rtl">

اگر جمله‌ی اعلام در متن بماند، مدل فقط همان regex را یاد می‌گیرد و بعد از ممنوعیت که این جمله نیست به کاری نمی‌آید. پس هر جمله‌ای که یک کلمه‌ی نشانه دارد از متن حذف می‌شود. این کار روی همه‌ی متن‌ها انجام می‌شود، مثبت و منفی و بعد از ممنوعیت، تا حذف شدن یک جمله خودش نشانه‌ی مثبت بودن نشود. نسخه‌ی اول فقط کلمه‌های خود regex را حذف می‌کند.

</div>

In [21]:
def mask(text, hints):
    sentences = re.split(r'(?<=[.!?])\s+|\n+', text)
    return ' '.join(s for s in sentences if not hints.search(s))

hints_v1 = re.compile(r'discount|free|reduced price|sample|exchange|honest|unbiased', re.I)
print(mask('Great sound. I got it at a discount for my honest review. Battery lasts long.', hints_v1))

Great sound. Battery lasts long.


<div dir="rtl">

برای پیدا کردن نشت، یک مدل خطی ساده روی متن‌های حذف‌شده آموزش می‌بیند و کلمه‌هایی که بیشترین وزن مثبت را گرفته‌اند نگاه می‌شوند. اگر تکه‌ای از جمله‌ی اعلام هنوز در متن مانده باشد، بالای این فهرست می‌آید. این مدل فقط برای همین بررسی است و انتخاب مدل در بخش بعد انجام می‌شود.

</div>

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

def top_words(texts, n=30):
    vec = TfidfVectorizer(min_df=5, ngram_range=(1, 2), sublinear_tf=True)
    X = vec.fit_transform(texts)
    clf = SGDClassifier(loss='log_loss', alpha=1e-5, class_weight='balanced', random_state=0)
    clf.fit(X, pool['disclosed'])
    words = vec.get_feature_names_out()
    return list(words[np.argsort(clf.coef_[0])[::-1][:n]])

top_words(pool['reviewText'].map(lambda t: mask(t, hints_v1)))

['this',
 'opinions',
 'my own',
 'this review',
 'my review',
 'please',
 'helpful',
 'own',
 'my reviews',
 'pretty',
 'or',
 'full price',
 'am',
 'just as',
 'we',
 'opinions are',
 'cable',
 'review helpful',
 'always',
 'kids',
 'my',
 'your',
 'pros',
 'quality',
 'this is',
 'has',
 'disclaimer',
 'husband',
 'tested',
 'of this']

<div dir="rtl">

در نسخه‌ی اول هنوز تکه‌هایی از جمله‌ی اعلام نشت کرده است: "opinions are my own"، "full price" و "disclaimer". پس فهرست نشانه‌ها با این کلمه‌ها و کلمه‌های هم‌خانواده‌شان بزرگ‌تر شد.

</div>

In [23]:
hints = re.compile(r'exchange|discount|free|honest|unbiased|sample|provided|complimentary|promotion|disclaim|disclos'
                   r'|received (?:this|the|it)|reduced|coupon|seller|vine|opinion|influenc|full price|my own|bias'
                   r'|compensat|paid|retail', re.I)
pool['masked'] = pool['reviewText'].map(lambda t: mask(t, hints))
top_words(pool['masked'])

['this',
 'please',
 'helpful',
 'my review',
 'this review',
 'my reviews',
 'or',
 'pretty',
 'cable',
 'review',
 'am',
 'test',
 'we',
 'just as',
 'review helpful',
 'this is',
 'kids',
 'guarantee',
 'overall',
 'your',
 'adapters',
 'pros',
 'husband',
 'not only',
 'you',
 'quality',
 'always',
 'vote',
 'has',
 'tested']

<div dir="rtl">

حالا هیچ کلمه‌ای از جمله‌ی اعلام در فهرست نیست. کلمه‌های بالا بیشتر درباره‌ی عادت نظردهی‌اند ("my reviews"، "review helpful"، "please"، "vote"، "tested") و چند کلمه‌ی نوع کالا هم هست (cable، adapters). اینکه این سبک بعد از ممنوعیت هم کار می‌کند یا نه، در بخش After the ban سنجیده می‌شود.

</div>

In [24]:
from sklearn.metrics import roc_auc_score

words = pool['masked'].str.split().str.len()
print('reviews with fewer than 5 words after masking, positives:', round((words[pool['disclosed']] < 5).mean(), 3),
      ' negatives:', round((words[~pool['disclosed']] < 5).mean(), 3))
print('AUC of review length, before masking:', round(roc_auc_score(pool['disclosed'], pool['reviewText'].str.split().str.len()), 3))
print('AUC of review length, after masking:', round(roc_auc_score(pool['disclosed'], words), 3))

reviews with fewer than 5 words after masking, positives: 0.002  negatives: 0.001


AUC of review length, before masking: 0.795
AUC of review length, after masking: 0.749


<div dir="rtl">

حذف جمله‌ها تقریباً هیچ نظری را خالی نکرده است (۰.۲٪ مثبت‌ها و ۰.۱٪ منفی‌ها). نظرهای تشویقی بلندترند و طول متن به‌تنهایی AUC برابر ۰.۷۴۹ می‌دهد. حذف جمله‌ها این عدد را از ۰.۷۹۵ پایین آورده، پس masking به نفع مثبت‌ها نشانه‌ای در طول متن نساخته است. این ۰.۷۴۹ کفی است که مدل‌های متنی در بخش بعد باید از آن بالاتر بروند.

</div>

In [25]:
%%time
train['masked'] = train['reviewText'].map(lambda t: mask(t, hints))

CPU times: user 1min 51s, sys: 128 ms, total: 1min 51s
Wall time: 1min 52s


## Model comparison

<div dir="rtl">

مدل‌ها فقط با cross-validation روی داده‌ی آموزش مقایسه می‌شوند و داده‌ی بعد از ممنوعیت تا بخش بعد دست‌نخورده می‌ماند. چند نکته برای اینکه مقایسه منصفانه و بدون نشت باشد:
- fold‌ها با GroupKFold بر اساس نظردهنده ساخته می‌شوند. همه‌ی نظرهای یک نفر یا در train‌اند یا در validation، تا مدل با حفظ کردن قالب نوشتن یک نفر نمره نگیرد.
- TF-IDF داخل pipeline است، پس کلمه‌ها و وزن IDF در هر fold فقط از train همان fold ساخته می‌شوند.
- همه‌ی مدل‌ها روی همان ۵ fold سنجیده می‌شوند و هر مدل چند مقدار برای مهم‌ترین پارامترش امتحان می‌کند.
- معیار اصلی average precision (AP) است. فقط ترتیب نظرها مهم است و AP بیشتر از AUC به بالای فهرست حساس است، یعنی همان‌جایی که صف بررسی تشکیل می‌شود. مقدار AP برای یک مدل تصادفی برابر سهم مثبت‌ها یعنی ۰.۱۹۸ است.

</div>

In [26]:
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score

y = pool['disclosed'].values
groups = pool['reviewerID'].values
results = []

def compare(features, name, model, grid, X):
    search = GridSearchCV(model, grid, cv=GroupKFold(5), scoring=['average_precision', 'roc_auc'],
                          refit=False, n_jobs=2)
    search.fit(X, y, groups=groups)
    cv = pd.DataFrame(search.cv_results_)
    best = cv.loc[cv['mean_test_average_precision'].idxmax()]
    results.append({'features': features, 'model': name, 'params': best['params'],
                    'AP': best['mean_test_average_precision'], 'AP std': best['std_test_average_precision'],
                    'AUC': best['mean_test_roc_auc'], 'fit time (s)': best['mean_fit_time']})
    return cv[['params', 'mean_test_average_precision', 'mean_test_roc_auc']].round(3)

<div dir="rtl">

baseline: فقط طول متن بعد از masking. این امتیاز آموزشی ندارد، پس مستقیم روی کل داده‌ی آموزش حساب شد.

</div>

In [27]:
length = pool['masked'].str.split().str.len()
results.append({'features': 'length only', 'model': 'baseline', 'params': {},
                'AP': average_precision_score(y, length), 'AP std': 0, 'AUC': roc_auc_score(y, length), 'fit time (s)': 0})
results[-1]

{'features': 'length only',
 'model': 'baseline',
 'params': {},
 'AP': 0.37794007954002956,
 'AP std': 0,
 'AUC': 0.7491437322257847,
 'fit time (s)': 0}

### TF-IDF models

<div dir="rtl">

ورودی این مدل‌ها TF-IDF یک‌کلمه‌ای و دوکلمه‌ای است. کلمه‌هایی که در کمتر از ۵ نظر آمده‌اند حذف می‌شوند و تعداد تکرار با لگاریتم نرم می‌شود (sublinear_tf) تا نظرهای بلند فقط با تکرار یک کلمه وزن زیادی نگیرند. random forest روی بیش از ۱۰۰ هزار ستون TF-IDF خیلی کند است و فقط روی بردارهای MiniLM امتحان شد.

</div>

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from lightgbm import LGBMClassifier

def tfidf():
    return TfidfVectorizer(min_df=5, ngram_range=(1, 2), sublinear_tf=True)

In [29]:
%%time
compare('TF-IDF', 'logistic regression', make_pipeline(tfidf(), LogisticRegression(solver='liblinear')),
        {'logisticregression__C': [1, 4, 16]}, pool['masked'])

CPU times: user 610 ms, sys: 369 ms, total: 979 ms
Wall time: 1min 8s


,params,mean_test_average_precision,mean_test_roc_auc
0,{'logisticregression__C': 1},0.612,0.862
1,{'logisticregression__C': 4},0.609,0.858
2,{'logisticregression__C': 16},0.591,0.847


In [30]:
%%time
compare('TF-IDF', 'linear SVM', make_pipeline(tfidf(), LinearSVC(random_state=0)),
        {'linearsvc__C': [0.1, 0.3, 1]}, pool['masked'])

CPU times: user 760 ms, sys: 334 ms, total: 1.09 s
Wall time: 1min 2s


,params,mean_test_average_precision,mean_test_roc_auc
0,{'linearsvc__C': 0.1},0.616,0.862
1,{'linearsvc__C': 0.3},0.611,0.857
2,{'linearsvc__C': 1},0.585,0.841


In [31]:
%%time
compare('TF-IDF', 'naive Bayes', make_pipeline(tfidf(), MultinomialNB()),
        {'multinomialnb__alpha': [0.03, 0.1, 0.3]}, pool['masked'])

CPU times: user 600 ms, sys: 348 ms, total: 947 ms
Wall time: 57.8 s


,params,mean_test_average_precision,mean_test_roc_auc
0,{'multinomialnb__alpha': 0.03},0.499,0.784
1,{'multinomialnb__alpha': 0.1},0.502,0.790
2,{'multinomialnb__alpha': 0.3},0.478,0.769


In [32]:
%%time
compare('TF-IDF', 'gradient boosting (LightGBM)',
        make_pipeline(tfidf(), LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1, n_jobs=1)),
        {'lgbmclassifier__num_leaves': [31]}, pool['masked'])

CPU times: user 1.45 s, sys: 332 ms, total: 1.78 s
Wall time: 8min 32s


,params,mean_test_average_precision,mean_test_roc_auc
0,{'lgbmclassifier__num_leaves': 31},0.612,0.856


### MiniLM embeddings

<div dir="rtl">

all-MiniLM-L6-v2 یک ترنسفورمر کوچک از پیش‌آموخته است که هر متن را به یک بردار ۳۸۴ بُعدی تبدیل می‌کند، و همین بردارها ورودی سه مدل کلاسیک‌اند. ساختن بردارها از برچسب‌ها استفاده نمی‌کند، پس یک بار برای کل داده‌ی آموزش ساخته شدند و نشتی ندارند. متن‌های بلندتر از ۲۵۶ توکن بریده می‌شوند.

</div>

In [33]:
%%time
from sentence_transformers import SentenceTransformer

minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
minilm.max_seq_length = 256
emb = minilm.encode(pool['masked'].tolist(), batch_size=64)
np.save(out + 'pool_minilm.npy', emb)
emb.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   7%|▋         | 7/103 [00:00<00:02, 36.62it/s]

Loading weights:  17%|█▋        | 18/103 [00:00<00:02, 33.54it/s]

Loading weights:  25%|██▌       | 26/103 [00:00<00:01, 43.97it/s]

Loading weights:  35%|███▍      | 36/103 [00:00<00:01, 50.67it/s]

Loading weights:  41%|████      | 42/103 [00:00<00:01, 43.18it/s]

... (6 outputs left out)

Loading weights:  96%|█████████▌| 99/103 [00:02<00:00, 40.67it/s]

Loading weights: 100%|██████████| 103/103 [00:02<00:00, 41.93it/s]

CPU times: user 26min 12s, sys: 2min 33s, total: 28min 46s
Wall time: 15min 49s


(42321, 384)

In [34]:
tokens = pool['masked'].map(lambda t: len(minilm.tokenizer.tokenize(t)))
print('share longer than 256 tokens, positives:', round((tokens[pool['disclosed']] > 256).mean(), 3),
      ' negatives:', round((tokens[~pool['disclosed']] > 256).mean(), 3))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (299 > 256). Running this sequence through the model will result in indexing errors


share longer than 256 tokens, positives: 0.427  negatives: 0.149


In [35]:
%%time
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

compare('MiniLM', 'logistic regression', make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
        {'logisticregression__C': [0.01, 0.1, 1]}, emb)

CPU times: user 167 ms, sys: 43.5 ms, total: 210 ms
Wall time: 7.95 s


,params,mean_test_average_precision,mean_test_roc_auc
0,{'logisticregression__C': 0.01},0.507,0.817
1,{'logisticregression__C': 0.1},0.509,0.818
2,{'logisticregression__C': 1},0.508,0.818


In [36]:
%%time
compare('MiniLM', 'random forest', RandomForestClassifier(n_estimators=300, n_jobs=1, random_state=0),
        {'min_samples_leaf': [1, 5]}, emb)

CPU times: user 1.86 s, sys: 466 ms, total: 2.33 s
Wall time: 15min 20s


,params,mean_test_average_precision,mean_test_roc_auc
0,{'min_samples_leaf': 1},0.436,0.759
1,{'min_samples_leaf': 5},0.463,0.784


In [37]:
%%time
compare('MiniLM', 'gradient boosting (LightGBM)', LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1, n_jobs=1),
        {'num_leaves': [15, 31]}, emb)

CPU times: user 367 ms, sys: 90.8 ms, total: 457 ms
Wall time: 1min 59s


,params,mean_test_average_precision,mean_test_roc_auc
0,{'num_leaves': 15},0.490,0.807
1,{'num_leaves': 31},0.497,0.810


### Fine-tuned MiniLM

<div dir="rtl">

برای مقایسه با یک مدل DL واقعی، MiniLM در یک نوت‌بوک جدا (bonus_minilm_finetune.ipynb) روی GPU کولب fine-tune می‌شود. برای اینکه مقایسه منصفانه باشد، همان متن‌های حذف‌شده و همان ۵ fold در یک فایل ذخیره می‌شوند و آن نوت‌بوک از همین فایل استفاده می‌کند.

</div>

In [38]:
pool['fold'] = -1
for k, (_, val_idx) in enumerate(GroupKFold(5).split(pool, y, groups)):
    pool.loc[val_idx, 'fold'] = k

pool[['masked', 'disclosed', 'reviewerID', 'fold']].to_csv(out + 'pool.csv.gz', index=False)
pool['fold'].value_counts().sort_index()

fold
0    8465
1    8464
2    8464
3    8464
4    8464
Name: count, dtype: int64

<div dir="rtl">

نتیجه‌ی آن نوت‌بوک (AP و AUC هر fold) از minilm_finetune_cv.csv خوانده می‌شود. چون foldها یکی‌اند، مدل fine-tune شده fold به fold با linear SVM هم مقایسه شد. زمان آموزش این مدل روی GPU است و با زمان بقیه‌ی مدل‌ها که روی CPU‌اند قابل مقایسه نیست.

</div>

In [39]:
finetune = pd.read_csv(out + 'minilm_finetune_cv.csv')
results.append({'features': 'MiniLM', 'model': 'fine-tuned (GPU)', 'params': {'epochs': 2, 'max_length': 512},
                'AP': finetune['AP'].mean(), 'AP std': finetune['AP'].std(ddof=0), 'AUC': finetune['AUC'].mean(),
                'fit time (s)': finetune['minutes'].mean() * 60})

svm_folds = []
for k in range(5):
    fit_part, val_part = pool[pool['fold'] != k], pool[pool['fold'] == k]
    svm = make_pipeline(tfidf(), LinearSVC(C=0.1, random_state=0)).fit(fit_part['masked'], fit_part['disclosed'])
    scores = svm.decision_function(val_part['masked'])
    svm_folds.append({'fold': k, 'SVM AP': average_precision_score(val_part['disclosed'], scores),
                      'SVM AUC': roc_auc_score(val_part['disclosed'], scores)})

paired = pd.DataFrame(svm_folds).merge(finetune[['fold', 'AP', 'AUC']], on='fold')
paired = paired.rename(columns={'AP': 'fine-tuned AP', 'AUC': 'fine-tuned AUC'})
paired['AP gain'] = paired['fine-tuned AP'] - paired['SVM AP']
paired.round(3)

,fold,SVM AP,SVM AUC,fine-tuned AP,fine-tuned AUC,AP gain
0,0,0.592,0.854,0.626,0.871,0.035
1,1,0.586,0.854,0.626,0.877,0.039
2,2,0.664,0.871,0.660,0.885,-0.004
3,3,0.604,0.861,0.616,0.874,0.012
4,4,0.632,0.871,0.650,0.881,0.018


### Comparison table

In [40]:
table = pd.DataFrame(results).sort_values('AP', ascending=False).reset_index(drop=True)
table.round(3)

,features,model,params,AP,AP std,AUC,fit time (s)
0,MiniLM,fine-tuned (GPU),"{'epochs': 2, 'max_length': 512}",0.636,0.016,0.878,445.341
1,TF-IDF,linear SVM,{'linearsvc__C': 0.1},0.616,0.029,0.862,6.398
2,TF-IDF,logistic regression,{'logisticregression__C': 1},0.612,0.029,0.862,7.005
3,TF-IDF,gradient boosting (LightGBM),{'lgbmclassifier__num_leaves': 31},0.612,0.023,0.856,170.895
4,MiniLM,logistic regression,{'logisticregression__C': 0.1},0.509,0.023,0.818,0.600
5,TF-IDF,naive Bayes,{'multinomialnb__alpha': 0.1},0.502,0.022,0.790,6.104
6,MiniLM,gradient boosting (LightGBM),{'num_leaves': 31},0.497,0.028,0.810,26.946
7,MiniLM,random forest,{'min_samples_leaf': 5},0.463,0.018,0.784,171.142
8,length only,baseline,{},0.378,0.000,0.749,0.000


In [41]:
table['name'] = table['features'] + ' + ' + table['model']
plot = table.sort_values('AP')
fig = px.bar(plot, x='AP', y='name', error_x='AP std', color='features', orientation='h',
             text=plot['AP'].round(3), hover_data=['AUC', 'params'],
             title='Cross-validated average precision (5 folds grouped by reviewer)')
fig.update_traces(textposition='inside', insidetextanchor='start')
fig.add_vline(x=y.mean(), line_dash='dash', annotation_text='random', annotation_position='bottom right')
fig.update_layout(yaxis_title='', height=450)
fig.show()

<div dir="rtl">

MiniLM fine-tune شده با AP برابر ۰.۶۳۶ و AUC برابر ۰.۸۷۸ بالای جدول است. فاصله‌اش با linear SVM کم است (۰.۰۲ در AP)، ولی در ۴ fold از ۵ fold بهتر است و AUC آن در هر ۵ fold بالاتر است. پس fine-tune کردن کمک کرده و فقط شانس fold نیست. بعد از آن سه مدل TF-IDF با AP بین ۰.۶۱۲ و ۰.۶۱۶ عملاً برابرند و همه از baseline طول متن (۰.۳۷۸) خیلی بهترند، یعنی چیزی بیشتر از «نظر بلند» یاد گرفته‌اند. اینکه gradient boosting از مدل‌های خطی بهتر نشده نشان می‌دهد سیگنال TF-IDF بیشتر از جمع اثر تک‌تک کلمه‌ها می‌آید.

بردارهای ثابت MiniLM، بدون fine-tune، حدود ۰.۱ پایین‌ترند. یک دلیلش بریده شدن متن است: ۴۳٪ نظرهای تشویقی از ۲۵۶ توکن بلندترند، در برابر ۱۵٪ بقیه، پس آخر نزدیک نیمی از این نظرها اصلاً به مدل نمی‌رسد. مدل fine-tune شده با ۵۱۲ توکن آموزش دید. Naive Bayes هم به‌خاطر فرض مستقل بودن کلمه‌ها عقب‌تر است.

</div>

### Final model

<div dir="rtl">

مدل کاری linear SVM با TF-IDF است، با اینکه MiniLM fine-tune شده در cross-validation کمی بهتر بود. سه دلیل:
- وزن هر کلمه در SVM مستقیم سهمش را در امتیاز نشان می‌دهد. در صف بررسی و داشبورد باید توضیح داد چرا یک نظر مشکوک است، و مدل fine-tune شده چنین وزن مستقیمی ندارد.
- SVM در چند ثانیه روی CPU به صدها هزار نظر امتیاز می‌دهد، در حالی که مدل fine-tune شده برای همین کار GPU لازم دارد.
- برتری مدل fine-tune شده (۰.۰۲ در AP) در برابر این دو هزینه کوچک است.

مدل fine-tune شده کنار گذاشته نمی‌شود و در بخش Fine-tuned MiniLM after the ban روی همان آزمون بعد از ممنوعیت با SVM مقایسه می‌شود. خروجی SVM احتمال نیست و فقط یک امتیاز است، ولی در این مسئله فقط ترتیب نظرها مهم است. مدل با C=0.1 روی کل داده‌ی آموزش fit می‌شود.

</div>

In [42]:
model = make_pipeline(tfidf(), LinearSVC(C=0.1, random_state=0))
model.fit(pool['masked'], y)

weights = pd.Series(model[1].coef_[0], index=model[0].get_feature_names_out())
top = pd.concat([weights.nlargest(20), weights.nsmallest(20)]).sort_values()
fig = px.bar(x=top.values, y=top.index, orientation='h', color=top.values > 0, height=800,
             color_discrete_map={True: '#D62728', False: '#1F77B4'},
             title='Words with the largest weights in the final model')
fig.update_layout(xaxis_title='weight', yaxis_title='', showlegend=False)
fig.show()

<div dir="rtl">

کلمه‌های مثبت بیشتر درباره‌ی خود نظرند، نه کالا: "helpful"، "please"، "my review"، "review helpful" و "vote"، مثل جمله‌ی "I hope this review was helpful, please vote". چند کلمه‌ی نوع کالا هم مانده است (cable، kids). کلمه‌های منفی نشانه‌ی خرید و استفاده‌ی واقعی‌اند: "bought"، "the price"، "months"، "return"، "customer service" و "update".

</div>

In [43]:
import joblib

joblib.dump(model, out + 'svm_model.joblib')
print('model file size (MB):', round(os.path.getsize(out + 'svm_model.joblib') / 1e6, 1))

model file size (MB): 4.6


## After the ban

<div dir="rtl">

مدل نهایی به همه‌ی نظرهای بعد از ممنوعیت امتیاز می‌دهد. این نظرها برچسب ندارند، پس مدل با سه آزمون بدون برچسب دستی سنجیده می‌شود: نظرهای بعدیِ نظردهنده‌هایی که قبلاً اعلام کرده بودند، سهم verified در امتیازهای بالا، و خود متن بالای فهرست. مدل هیچ نظری از بعد از ممنوعیت ندیده است.

</div>

In [44]:
post = train[~train['pre_ban']].copy()
post['score'] = model.decision_function(post['masked'])

seen = set(incentivized) - held_out
maybe_reviewers = set(pre.loc[pre['maybe'], 'reviewerID'])
post['group'] = np.select([post['reviewerID'].isin(held_out), post['reviewerID'].isin(seen),
                           post['reviewerID'].isin(maybe_reviewers)],
                          ['held out', 'seen in training', 'possible disclosure'], 'others')
post['group'].value_counts()

group
others                 453019
seen in training         3089
possible disclosure      2250
held out                 1292
Name: count, dtype: int64

### Held-out reviewers

<div dir="rtl">

آزمون اصلی گروه held out است: ۳۰٪ نظردهنده‌هایی که قبل از ممنوعیت اعلام کرده بودند و هیچ نظری از آن‌ها در آموزش نبود. اگر این افراد بعد از ممنوعیت هم تشویقی ولی بدون اعلام نوشته باشند، مدل باید نظرهایشان را بالاتر از بقیه بگذارد. هر گروه با others مقایسه شد، یعنی کسانی که قبل از ممنوعیت هیچ اعلامی نداشتند. AUC داخل نظرهای ۵ ستاره هم حساب شد تا معلوم شود مدل فقط نظرهای مثبت را بالا نمی‌برد.

</div>

In [45]:
rows = []
others = post['group'] == 'others'
for g in ['held out', 'seen in training', 'possible disclosure']:
    part = post[(post['group'] == g) | others]
    five = part['overall'] == 5
    rows.append({'group': g, 'reviews': (part['group'] == g).sum(),
                 'AUC': roc_auc_score(part['group'] == g, part['score']),
                 'AUC (5 stars only)': roc_auc_score(part.loc[five, 'group'] == g, part.loc[five, 'score'])})
pd.DataFrame(rows).round(3)

,group,reviews,AUC,AUC (5 stars only)
0,held out,1292,0.714,0.723
1,seen in training,3089,0.723,0.717
2,possible disclosure,2250,0.663,0.676


In [46]:
test = post[post['group'].isin(['held out', 'others'])]
base = (test['group'] == 'held out').mean()
for k in [100, 1000, 5000]:
    held = test.nlargest(k, 'score').query("group == 'held out'")
    share = len(held) / k
    print(f'top {k}: {share:.1%} from held-out reviewers ({share / base:.0f}x the base rate of {base:.2%}),',
          held['reviewerID'].nunique(), 'different people')

top 100: 22.0% from held-out reviewers (77x the base rate of 0.28%), 6 different people
top 1000: 6.8% from held-out reviewers (24x the base rate of 0.28%), 36 different people


top 5000: 3.0% from held-out reviewers (11x the base rate of 0.28%), 74 different people


<div dir="rtl">

بعضی نظردهنده‌ها بعد از ممنوعیت خیلی پرکارند و می‌توانند به‌تنهایی بالای فهرست را پر کنند. برای اینکه نتیجه به چند نفر وابسته نباشد، AUC در سطح نظردهنده هم حساب شد: هر نفر یک امتیاز دارد که میانگین امتیاز نظرهایش است.

</div>

In [47]:
by_reviewer = test.groupby('reviewerID').agg(score=('score', 'mean'), held=('group', lambda g: (g == 'held out').any()))
print('held-out reviewers with post-ban reviews:', by_reviewer['held'].sum())
print('reviewer-level AUC:', round(roc_auc_score(by_reviewer['held'], by_reviewer['score']), 3))

held-out reviewers with post-ban reviews: 476
reviewer-level AUC: 0.732


In [48]:
bins = np.linspace(test['score'].min(), test['score'].max(), 51)
density = pd.DataFrame({g: np.histogram(test.loc[test['group'] == g, 'score'], bins=bins, density=True)[0]
                        for g in ['held out', 'others']})
density['score'] = (bins[:-1] + bins[1:]) / 2
fig = px.line(density, x='score', y=['held out', 'others'],
              color_discrete_map={'held out': '#D62728', 'others': '#7F7F7F'},
              title='Scores of post-ban reviews: held-out former disclosers vs others')
fig.update_layout(xaxis_title='model score', yaxis_title='density', legend_title='')
fig.show()

<div dir="rtl">

AUC نظرهای held out برابر ۰.۷۱۴ است و داخل نظرهای ۵ ستاره ۰.۷۲۳. پس مدل نظرهای این افراد را بهتر از تصادف تشخیص می‌دهد و این فقط به‌خاطر مثبت بودن لحن نیست. در ۱۰۰۰ نظر بالای فهرست، سهم این افراد ۲۴ برابر سهم پایه است و این نظرها از ۳۶ نفر مختلف‌اند. عدد ۱۰۰ نظر اول (۷۷ برابر) را باید با احتیاط خواند، چون ۲۲ نظرش فقط از ۶ نفر است. AUC در سطح نظردهنده (۰.۷۳۲ برای ۴۷۶ نفر) نشان می‌دهد نتیجه به چند نفر پرکار وابسته نیست. سیگنال در بالای فهرست قوی است و در کل فهرست متوسط، که با کاربرد صف بررسی جور است. AUC گروه seen in training (۰.۷۲۳) تقریباً با held out برابر است، پس مدل قالب نوشتن آدم‌ها را حفظ نکرده و بیشتر یک سبک کلی را یاد گرفته است. گروه possible disclosure هم (کسانی که فقط یکی از عبارت‌های دیگر اعلام را داشتند) با AUC ۰.۶۶۳ بالاتر از بقیه است.

</div>

### Verified purchases

<div dir="rtl">

نظر تشویقی معمولاً verified نیست، چون کالا با قیمت معمول خریده نشده است. پس اگر مدل نظرهای تشویقی پنهان را پیدا کند، سهم verified در امتیازهای بالا باید کم شود. این شاخص دوطرفه است: نظرهای برنامه‌ی Vine بعد از ممنوعیت مجازند و verified نیستند، و در داده‌ی ما از بقیه جدا نمی‌شوند. از طرف دیگر، طرحی که پول کالا را بیرون از آمازون پس می‌دهد نظر verified می‌سازد. برای همین این آزمون فقط کمکی است.

</div>

In [49]:
post['decile'] = pd.qcut(post['score'], 10, labels=range(1, 11))
by_decile = post.groupby('decile', observed=True).agg(verified=('verified', 'mean'), stars=('overall', 'mean'))
top1000 = post.nlargest(1000, 'score')
print('top 1000 reviews: verified', round(top1000['verified'].mean(), 3), ' mean rating', round(top1000['overall'].mean(), 2))
by_decile.round(3).T

top 1000 reviews: verified 0.682  mean rating 4.65


decile,1,2,3,4,5,6,7,8,9,10
verified,0.885,0.897,0.898,0.902,0.906,0.900,0.898,0.893,0.876,0.821
stars,3.269,3.588,3.760,3.883,3.977,4.067,4.149,4.253,4.344,4.460


In [50]:
fig = px.bar(by_decile.reset_index(), x='decile', y='verified', text=by_decile['verified'].round(2),
             title='Share of verified purchases by score decile (post-ban reviews)')
fig.add_hline(y=top1000['verified'].mean(), line_dash='dash', annotation_text='top 1000 reviews')
fig.update_layout(xaxis_title='score decile (10 = most suspicious)', yaxis_title='verified share', yaxis_range=[0.6, 0.95],
                  xaxis_dtick=1)
fig.show()

<div dir="rtl">

سهم verified تا دهک نهم تقریباً ثابت است (بین ۰.۸۸ و ۰.۹۱). در دهک آخر به ۰.۸۲ و در ۱۰۰۰ نظر اول به ۰.۶۸ می‌رسد. پس این آزمون هم همان نتیجه را می‌دهد: سیگنال در بالای فهرست متمرکز است. میانگین امتیاز ستاره‌ای هم با دهک بالا می‌رود (از ۳.۲۷ به ۴.۴۶). این انتظار می‌رفت، چون نظرهای تشویقی بیشتر مثبت‌اند، و AUC داخل نظرهای ۵ ستاره نشان داد مدل فقط همین را یاد نگرفته است.

</div>

### Top of the list

<div dir="rtl">

ده نظر با بیشترین امتیاز بعد از ممنوعیت، فقط برای دیدن. برای کوتاه شدن، ۲۰۰ کاراکتر اول و آخر هر متن آمده است.

</div>

In [51]:
for _, r in post.nlargest(10, 'score').iterrows():
    text = r['reviewText'].replace('\n', ' ')
    print(f"score {r['score']:.2f} | {r['overall']} stars | verified {r['verified']} | {r['group']}")
    print(text[:200], '...', text[-200:], '\n')

score 1.53 | 5 stars | verified False | others
So they come as a 3 pack! They are very short syncing cables. They work perfect for what I wanted them for. I like them when I am charging my phone with my computer so it is easier for me when the cor ... only does it meet all expectations but its a product thats well made (designed, formulated, composed, etc) and lives up to the product details and description. It exceeds normal consumer expectations. 

score 1.43 | 5 stars | verified False | others
Camera is easy to set up and works great. Video is good! I like this camera and would recommend.  I am under no obligation to provide a positive review and receive no incentives or rewards for doing s ... only does it meet all expectations but its a product thats well made (designed, formulated, composed, etc) and lives up to the product details and description. It exceeds normal consumer expectations. 

score 1.36 | 5 stars | verified False | held out
This MP3 player is very well made and wor

<div dir="rtl">

چند جمله در این نظرها، روی محصول‌های مختلف، عیناً تکرار شده است. این جمله‌ها در کل داده شمرده شدند.

</div>

In [52]:
phrases = ['would i recommend this product to other people', 'exceeds normal consumer expectations',
           'under no obligation to provide a positive review']
low = train['reviewText'].str.lower()
rows = []
for phrase in phrases:
    m = low.str.contains(phrase, regex=False)
    after = m & ~train['pre_ban']
    rows.append({'phrase': phrase, 'before the ban': (m & train['pre_ban']).sum(),
                 'disclosed, before the ban': (m & train['disclosed']).sum(), 'after the ban': after.sum(),
                 'reviewers after the ban': train.loc[after, 'reviewerID'].nunique(),
                 'products after the ban': train.loc[after, 'asin'].nunique()})
pd.DataFrame(rows)

,phrase,before the ban,"disclosed, before the ban",after the ban,reviewers after the ban,products after the ban
0,would i recommend this product to other people,1,1,17,1,17
1,exceeds normal consumer expectations,21,0,3,1,3
2,under no obligation to provide a positive review,253,120,6,2,6


<div dir="rtl">

این جمله‌ها بعد از ممنوعیت هر کدام فقط از یک یا دو نفر آمده‌اند ولی روی محصول‌های مختلف تکرار شده‌اند، مثلاً جمله‌ی اول ۱۷ بار از یک نفر روی ۱۷ محصول. جمله‌ی "under no obligation to provide a positive review" قبل از ممنوعیت ۲۵۳ بار آمده بود و نزدیک نصف این موارد (۱۲۰) جمله‌ی اعلام هم داشتند. بعد از ممنوعیت همین جمله بدون اعلام آمده است.

</div>

## Review rings

<div dir="rtl">

آزمون دوم کاملاً مستقل از متن است. روی محصول‌های پرفروش، دو نظردهنده ممکن است تصادفی چند محصول مشترک داشته باشند، ولی روی محصول‌های کم‌نظر این تصادف بعید است. پس جفت نظردهنده‌هایی پیدا شدند که روی تعداد زیادی محصول کم‌نظر (۱۰۰ نظر یا کمتر در کل داده) مشترکاً نظر داده‌اند. ماتریس sparse نظردهنده × محصول ساخته می‌شود و ضرب آن در ترانهاده‌ی خودش، تعداد محصول‌های مشترک هر جفت را می‌دهد. فقط نظردهنده‌هایی آمده‌اند که دست‌کم ۳ نظر روی این محصول‌ها دارند.

</div>

In [53]:
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components

size = train.groupby('asin')['asin'].transform('size')
small = train[size <= 100]
counts = small['reviewerID'].value_counts()
small = small[small['reviewerID'].isin(counts[counts >= 3].index)]

reviewers = small['reviewerID'].astype('category')
items = small['asin'].astype('category')
R = sp.csr_matrix((np.ones(len(small)), (reviewers.cat.codes, items.cat.codes)))
R.data[:] = 1
shared = (R @ R.T).tocoo()
upper = shared.row < shared.col
print('reviewers:', R.shape[0], ' small products:', R.shape[1])

for k in [3, 4, 5, 6]:
    pairs = upper & (shared.data >= k)
    members = np.unique(np.r_[shared.row[pairs], shared.col[pairs]])
    print(f'pairs sharing at least {k} small products: {pairs.sum()}, reviewers in them: {len(members)}')

reviewers: 73281  small products: 74346
pairs sharing at least 3 small products: 9947, reviewers in them: 1653
pairs sharing at least 4 small products: 1402, reviewers in them: 461
pairs sharing at least 5 small products: 546, reviewers in them: 205
pairs sharing at least 6 small products: 316, reviewers in them: 119


<div dir="rtl">

با ۳ محصول مشترک نزدیک ۱۰ هزار جفت پیدا می‌شود که برای نظردهنده‌های پرکار می‌تواند تصادفی باشد. آستانه‌ی ۵ سخت‌گیرانه است و فقط ۲۰۵ نفر می‌مانند. جفت‌ها به هم وصل شدند تا گروه‌ها (حلقه‌ها) معلوم شوند.

</div>

In [54]:
pairs = upper & (shared.data >= 5)
n = R.shape[0]
graph = sp.coo_matrix((np.ones(pairs.sum()), (shared.row[pairs], shared.col[pairs])), shape=(n, n))
_, ring_id = connected_components(graph, directed=False)

in_ring = np.zeros(n, dtype=bool)
in_ring[shared.row[pairs]] = True
in_ring[shared.col[pairs]] = True
rings = pd.DataFrame({'reviewerID': reviewers.cat.categories[in_ring], 'ring': ring_id[in_ring]})
rings['ring_size'] = rings.groupby('ring')['ring'].transform('size')
print('reviewers in rings:', len(rings), ' rings:', rings['ring'].nunique())
rings.drop_duplicates('ring')['ring_size'].value_counts().sort_index()

reviewers in rings: 205  rings: 11


ring_size
2      5
3      1
4      1
5      1
12     2
159    1
Name: count, dtype: int64

In [55]:
disclosers = set(train.loc[train['disclosed'], 'reviewerID'])
active = pd.Series(reviewers.cat.categories)
others_active = active[~active.isin(rings['reviewerID'])]

def describe(people):
    their = train[train['reviewerID'].isin(people)]
    return [their['overall'].mean(), their['verified'].mean(), people.isin(disclosers).mean()]

pd.DataFrame({'ring members': describe(rings['reviewerID']), 'other active reviewers': describe(others_active)},
             index=['mean rating', 'verified share', 'share with a disclosed review']).round(3)

,ring members,other active reviewers
mean rating,4.490,4.132
verified share,0.147,0.836
share with a disclosed review,0.127,0.044


<div dir="rtl">

بزرگ‌ترین حلقه ۱۵۹ عضو دارد. در این شبکه هر نقطه یک نظردهنده است و هر خط یعنی آن دو نفر دست‌کم ۵ محصول کم‌نظر مشترک دارند. نقطه‌های قرمز دست‌کم یک نظر اعلام‌شده دارند.

</div>

In [56]:
import networkx as nx

biggest = rings.loc[rings['ring_size'] == rings['ring_size'].max(), 'reviewerID']
codes = set(reviewers.cat.categories.get_indexer(biggest))
G = nx.Graph([(a, b) for a, b in zip(shared.row[pairs], shared.col[pairs]) if a in codes])
pos = nx.spring_layout(G, seed=0)

edge_x, edge_y = [], []
for a, b in G.edges():
    edge_x += [pos[a][0], pos[b][0], None]
    edge_y += [pos[a][1], pos[b][1], None]
nodes = list(G.nodes())
red = [reviewers.cat.categories[v] in disclosers for v in nodes]

fig = go.Figure([go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=0.5, color='#BBBBBB'), hoverinfo='skip'),
                 go.Scatter(x=[pos[v][0] for v in nodes], y=[pos[v][1] for v in nodes], mode='markers',
                            marker=dict(size=8, color=['#D62728' if r else '#1F77B4' for r in red]),
                            text=[f'{G.degree(v)} connections' for v in nodes], hoverinfo='text')])
fig.update_layout(title=f'Largest review ring: {len(nodes)} reviewers, {G.number_of_edges()} pairs',
                  showlegend=False, height=650, xaxis_visible=False, yaxis_visible=False)
fig.show()

<div dir="rtl">

اعضای حلقه‌ها خیلی بیشتر از بقیه‌ی نظردهنده‌های فعال روی محصول‌های کم‌نظر ۵ ستاره می‌دهند، خیلی کمتر verified دارند و چند برابر بیشتر سابقه‌ی اعلام دارند. حالا آزمون مستقل: آیا مدل متنی به نظرهای اعضای حلقه امتیاز بالاتری می‌دهد؟ فقط نظرهای بعد از ممنوعیتِ اعضایی آمده‌اند که هیچ‌وقت اعلامی نداشته‌اند (گروه others). اگر نظرهای قبل از ممنوعیت این افراد در داده‌ی آموزش بوده، به‌عنوان منفی بوده، پس این نشت فقط می‌تواند نتیجه را بدتر کند.

</div>

In [57]:
check = post[post['group'] == 'others'].copy()
check['ring'] = check['reviewerID'].isin(rings['reviewerID'])
five = check['overall'] == 5
print('post-ban reviews of ring members:', check['ring'].sum())
print('AUC, ring members vs all others:', round(roc_auc_score(check['ring'], check['score']), 3))
print('AUC, 5-star reviews only:', round(roc_auc_score(check.loc[five, 'ring'], check.loc[five, 'score']), 3))

similar = check[check['reviewerID'].isin(active)]
print('AUC, vs other reviewers with 3+ small-product reviews:', round(roc_auc_score(similar['ring'], similar['score']), 3))

post-ban reviews of ring members: 2356
AUC, ring members vs all others: 0.761
AUC, 5-star reviews only: 0.743


AUC, vs other reviewers with 3+ small-product reviews: 0.729


<div dir="rtl">

اعضای حلقه‌ها میانگین ۴.۴۹ ستاره می‌دهند و فقط ۱۴.۷٪ نظرهایشان verified است، در برابر ۴.۱۳ و ۸۳.۶٪ برای بقیه‌ی نظردهنده‌های فعال روی محصول‌های کم‌نظر. ۱۲.۷٪ اعضا دست‌کم یک نظر اعلام‌شده دارند، در برابر ۴.۴٪ بقیه. مدل متنی، که هیچ اطلاعی از این شبکه ندارد، به نظرهای بعد از ممنوعیتِ اعضایی که هرگز اعلام نکرده‌اند امتیاز بالاتری می‌دهد: AUC برابر ۰.۷۶۱، و ۰.۷۲۹ در مقایسه با نظردهنده‌های پرکار مشابه. دو روش مستقل، یکی از روی متن و یکی از روی رفتار، همان افراد را مشکوک نشان می‌دهند. ولی عضویت در حلقه هم اثبات نیست: آستانه‌ی ۵ محصول با آزمون آماری انتخاب نشده و ممکن است چند نظردهنده‌ی پرکار با سلیقه‌ی مشابه هم وارد شده باشند.

</div>

## Fine-tuned MiniLM after the ban

<div dir="rtl">

مدل fine-tune شده هم باید روی آزمون بعد از ممنوعیت سنجیده شود، نه فقط با cross-validation. امتیاز دادن به همه‌ی ۴۶۰ هزار نظر با این مدل GPU و وقت زیادی می‌خواهد، پس یک نمونه ساخته شد: همه‌ی نظرهای گروه held out، همه‌ی نظرهای اعضای حلقه‌ها در گروه others، و ۵۰ هزار نظر تصادفی دیگر از others. نوت‌بوک bonus_minilm_finetune.ipynb به این نمونه امتیاز می‌دهد و امتیاز SVM هم کنار هر نظر ذخیره می‌شود تا دو مدل روی نظرهای یکسان مقایسه شوند.

</div>

In [58]:
check_ids = check.index[check['ring']]
random_others = post[post['group'] == 'others'].drop(check_ids).sample(50000, random_state=0).index
sample_ids = post.index[post['group'] == 'held out'].union(check_ids).union(random_others)

sample = post.loc[sample_ids, ['masked', 'group', 'overall', 'score']].rename(columns={'score': 'svm_score'})
sample['ring'] = sample.index.isin(check_ids)
sample.index.name = 'row'
sample.to_csv(out + 'post_sample.csv.gz')
sample['group'].value_counts()

group
others      52356
held out     1292
Name: count, dtype: int64

<div dir="rtl">

امتیازهای مدل fine-tune شده از post_sample_finetune.csv خوانده می‌شوند. هر دو مدل روی همان نظرها با دو آزمون سنجیده می‌شوند: نظردهنده‌های held out و اعضای حلقه‌ها، هر کدام در برابر ۵۰ هزار نظر تصادفی. AUC در سطح نظردهنده هم آمده، چون نظرهای حلقه‌ها از ۱۴۴ نفر است و چند نفرشان خیلی پرکارند.

</div>

In [59]:
from scipy.stats import spearmanr

scored = sample.join(pd.read_csv(out + 'post_sample_finetune.csv', index_col='row'))
scored['reviewerID'] = post.loc[scored.index, 'reviewerID']
rest = scored[(scored['group'] == 'others') & ~scored['ring']]

rows = []
for test_name, part in [('held out', scored[scored['group'] == 'held out']), ('ring members', scored[scored['ring']])]:
    both = pd.concat([part, rest])
    target = both.index.isin(part.index)
    five = (both['overall'] == 5).values
    for model_name, col in [('SVM', 'svm_score'), ('fine-tuned MiniLM', 'finetune_score')]:
        person = both.assign(target=target).groupby('reviewerID').agg(target=('target', 'any'), score=(col, 'mean'))
        rows.append({'test': test_name, 'model': model_name, 'AUC': roc_auc_score(target, both[col]),
                     'AUC (5 stars only)': roc_auc_score(target[five], both.loc[five, col]),
                     'reviewer-level AUC': roc_auc_score(person['target'], person['score'])})

print('Spearman between the two scores:', round(spearmanr(scored['svm_score'], scored['finetune_score']).correlation, 3))
pd.DataFrame(rows).pivot(index='test', columns='model').round(3)

Spearman between the two scores: 0.717


AUC                   AUC (5 stars only)                    \
model           SVM fine-tuned MiniLM                SVM fine-tuned MiniLM   
test                                                                         
held out      0.716             0.722              0.724             0.728   
ring members  0.762             0.803              0.742             0.775   

             reviewer-level AUC                    
model                       SVM fine-tuned MiniLM  
test                                               
held out                  0.710             0.718  
ring members              0.829             0.868

<div dir="rtl">

روی آزمون اصلی، یعنی held out، دو مدل تقریباً برابرند: AUC برابر ۰.۷۱۶ و ۰.۷۲۲، و در سطح نظردهنده ۰.۷۱۰ و ۰.۷۱۸. پس برتری مدل fine-tune شده در cross-validation بعد از ممنوعیت خیلی کوچک می‌ماند. روی اعضای حلقه‌ها مدل fine-tune شده بهتر است (۰.۸۰۳ در برابر ۰.۷۶۲، و در سطح نظردهنده ۰.۸۶۸ در برابر ۰.۸۲۹)، ولی این آزمون فقط ۱۴۴ نفر دارد. همبستگی Spearman دو امتیاز ۰.۷۱۷ است، یعنی دو مدل تا حدی چیزهای متفاوتی می‌بینند و ترکیبشان می‌تواند قدم بعدی باشد. SVM مدل کاری می‌ماند، چون روی آزمون اصلی همان‌قدر خوب است و قابل توضیح و سبک است.

</div>

## Business output

<div dir="rtl">

خروجی کار برای فروشگاه سه چیز است: صف بررسی نظرها، فهرست محصول‌ها و برندهایی که نظر مشکوکشان غیرعادی زیاد است، و امتیاز اصلاح‌شده‌ی محصول‌ها. ۱٪ نظرهای بعد از ممنوعیت که بیشترین امتیاز را دارند علامت می‌خورند. این مرز از آزمون held out آمده: در ۵ هزار نظر اول (حدود ۱٪) سهم آن افراد هنوز ۱۱ برابر سهم پایه بود. امتیاز مدل حکم نیست، پس این نظرها برای بررسی و کم کردن وزن‌اند، نه حذف خودکار.

</div>

In [60]:
post['flagged'] = post['score'].rank(pct=True) > 0.99
post['brand'] = post['asin'].map(products.set_index('asin')['brand']).str.strip().str.lower()
print('flagged reviews:', post['flagged'].sum())

flagged reviews: 4597


### Products

<div dir="rtl">

اگر نظرهای علامت‌خورده بین محصول‌ها تصادفی پخش شده باشند، تعداد علامت‌خورده‌های محصولی با n نظر از توزیع دوجمله‌ای با احتمال ۱٪ می‌آید. p-value احتمال دیدن این تعداد یا بیشتر است. چون هزاران محصول آزمون می‌شوند، مرز سخت‌گیرانه‌ی ۰.۰۰۱ انتخاب شد. امتیاز اصلاح‌شده‌ی هر محصول میانگین نظرهای علامت‌نخورده‌ی آن است.

</div>

In [61]:
from scipy.stats import binom

per_product = post.groupby('asin').agg(reviews=('overall', 'size'), flagged=('flagged', 'sum'), rating=('overall', 'mean'))
per_product['adjusted'] = post[~post['flagged']].groupby('asin')['overall'].mean()
per_product['flagged_share'] = per_product['flagged'] / per_product['reviews']
per_product['p_value'] = binom.sf(per_product['flagged'] - 1, per_product['reviews'], 0.01)
per_product = per_product.join(products.set_index('asin')[['brand', 'title']])

tested = per_product[per_product['reviews'] >= 10]
watch = tested[tested['p_value'] < 0.001].sort_values('flagged', ascending=False)
print('products with 10+ post-ban reviews:', len(tested), ' with unusually many flagged reviews:', len(watch))
print('expected by chance at p < 0.001:', round(len(tested) * 0.001))
print('flagged reviews on these products:', watch['flagged'].sum(), 'of', post['flagged'].sum())

show = watch.head(15).copy()
show['title'] = show['title'].str[:50]
show[['brand', 'title', 'reviews', 'flagged', 'rating', 'adjusted']].round(2)

products with 10+ post-ban reviews: 9345  with unusually many flagged reviews: 68
expected by chance at p < 0.001: 9
flagged reviews on these products: 362 of 4597


,brand,title,reviews,flagged,rating,adjusted
asin,,,,,,
B00DIF2BO2,Cheetah,Cheetah Mounts APFMSB TV Wall Mount Bracket fo...,448,19,3.99,3.96
B0012S4APK,Cheetah,"Cheetah APTMM2B TV Wall Mount for 20-70"" TVs u...",315,12,3.90,3.87
B018E0FTTO,ODRVM,Action Camera HD 1080P Underwater Sports Camer...,56,10,4.52,4.48
B01DLWBPCA,Old Shark,"OldShark Dash Cam, 3"" 1080P HD Car Recorder 17...",146,10,4.01,3.96
B017IS057O,Albott,Albott 70 Inch Digital SLR Camera Aluminum Tra...,50,9,3.96,3.76
B019JSE7PC,Nucleus,Nucleus Anywhere Intercom with Amazon Alexa (N...,64,9,3.50,3.38
B003U46SWC,Fortress Mount,"Parent tv mount (40""-75"" (2016))",46,9,4.24,4.08
B00OHQFL0W,AudioAffairs,AudioAffairs Bluetooth Plug Radio,26,9,3.92,3.82
B013LKLS2E,Huawei,Huawei Watch Stainless Steel with Stainless St...,164,9,3.91,3.88


<div dir="rtl">

از ۹٬۳۴۵ محصولی که دست‌کم ۱۰ نظر بعد از ممنوعیت دارند، ۶۸ محصول نظر علامت‌خورده‌ی غیرعادی زیاد دارند، در حالی که به‌طور تصادفی حدود ۹ مورد انتظار می‌رود. ۳۶۲ نظر از ۴٬۵۹۷ نظر علامت‌خورده روی همین محصول‌هاست. این فهرست نقطه‌ی شروع تیم بررسی است. دو پایه‌ی دیواری تلویزیون از برند Cheetah با ۱۹ و ۱۲ نظر علامت‌خورده بالای فهرست‌اند.

</div>

<div dir="rtl">

برای دیدن اثر روی امتیاز محصول‌ها، حذف نظرهای علامت‌خورده با چند مرز مختلف تکرار شد. فقط محصول‌هایی که دست‌کم ۳۰ نظر بعد از ممنوعیت دارند آمده‌اند تا میانگینشان پایدار باشد.

</div>

In [62]:
from scipy.stats import spearmanr

big = per_product[per_product['reviews'] >= 30]
rows = []
for k in [0.005, 0.01, 0.02, 0.05]:
    keep_reviews = post['score'].rank(pct=True) <= 1 - k
    adjusted = post[keep_reviews].groupby('asin')['overall'].mean().reindex(big.index)
    drop = big['rating'] - adjusted
    rows.append({'flagged share': k, 'products': len(big), 'drop of 0.1+ stars': (drop >= 0.1).sum(),
                 'largest drop': drop.max(), 'Spearman with original': spearmanr(big['rating'], adjusted).correlation,
                 'left the top 100': len(set(big['rating'].nlargest(100).index) - set(adjusted.nlargest(100).index))})
pd.DataFrame(rows).round(3)

,flagged share,products,drop of 0.1+ stars,largest drop,Spearman with original,left the top 100
0,0.005,2795,0,0.093,1.000,1
1,0.010,2795,6,0.204,0.999,3
2,0.020,2795,17,0.269,0.998,5
3,0.050,2795,100,0.382,0.996,7


<div dir="rtl">

اثر روی امتیاز محصول‌ها کوچک است. با مرز ۱٪، از ۲٬۷۹۵ محصول فقط ۶ محصول ۰.۱ ستاره یا بیشتر افت می‌کنند، بیشترین افت ۰.۲۰ ستاره است و ترتیب محصول‌ها تقریباً عوض نمی‌شود (Spearman برابر ۰.۹۹۹). حتی با مرز ۵٪ فقط ۱۰۰ محصول ۰.۱ ستاره افت می‌کنند. پس نظرهای تشویقی پنهانی که این مدل بعد از ممنوعیت پیدا می‌کند آن‌قدر زیاد نیستند که امتیاز کلی محصول‌ها را جابه‌جا کنند. ارزش کار در این دوره صف بررسی و فهرست محصول‌ها و برندهای مشکوک است، نه اصلاح همه‌ی امتیازها. قبل از ممنوعیت وضع فرق داشت: نظرهای اعلام‌شده ۰.۵ ستاره بالاتر از بقیه‌ی نظرهای همان محصول بودند.

</div>

### Brands

<div dir="rtl">

سهم نظرهای علامت‌خورده برای برندهایی که دست‌کم ۳۰۰ نظر بعد از ممنوعیت دارند.

</div>

In [63]:
brands = post.groupby('brand').agg(reviews=('overall', 'size'), flagged_share=('flagged', 'mean'), rating=('overall', 'mean'))
brands['adjusted'] = post[~post['flagged']].groupby('brand')['overall'].mean()
brands = brands[brands['reviews'] >= 300].sort_values('flagged_share', ascending=False)
print('brands:', len(brands), ' brands above 3x the overall 1%:', (brands['flagged_share'] > 0.03).sum())

top = brands.head(20).reset_index().sort_values('flagged_share')
fig = px.bar(top, x='flagged_share', y='brand', orientation='h', text=top['flagged_share'].round(3),
             hover_data=['reviews', 'rating', 'adjusted'],
             title='Brands with the highest share of flagged reviews (300+ post-ban reviews)')
fig.add_vline(x=0.01, line_dash='dash', annotation_text='overall 1%')
fig.update_layout(xaxis_title='share of flagged reviews', yaxis_title='', xaxis_tickformat='.0%', height=600)
fig.show()

brands: 263  brands above 3x the overall 1%: 9


<div dir="rtl">

SOLO با ۱۳.۲٪ خیلی جلوتر از بقیه است و ۹ برند بیش از ۳ برابر سهم کلی ۱٪ دارند. چند برند بالای فهرست کیف و قاب می‌فروشند (SOLO، Lowepro، OtterBox و Evecase). چون چند کلمه‌ی نوع کالا هنوز در مدل هست، بخشی از این فاصله ممکن است به نوع کالا برگردد و نه به خود برند.

</div>

### Files for the dashboard

<div dir="rtl">

جدول‌های این نوت‌بوک برای داشبورد ذخیره می‌شوند. مدل SVM قبلاً در svm_model.joblib ذخیره شد. برای اینکه داشبورد امتیاز یک متن جدید را به صدک تبدیل کند، صدک‌های امتیاز نظرهای بعد از ممنوعیت هم ذخیره می‌شوند.

</div>

In [64]:
per_product.to_csv(out + 'products.csv')
brands.to_csv(out + 'brands.csv')
monthly.to_csv(out + 'monthly.csv')
rings.to_csv(out + 'rings.csv', index=False)

queue = post[post['flagged']].sort_values('score', ascending=False)
queue[['asin', 'brand', 'reviewerID', 'overall', 'verified', 'score', 'reviewText']].to_csv(out + 'review_queue.csv.gz', index=False)
np.save(out + 'score_percentiles.npy', np.percentile(post['score'], np.arange(101)))
sorted(os.listdir(out))

['brands.csv',
 'minilm_finetune_cv.csv',
 'monthly.csv',
 'pool.csv.gz',
 'pool_minilm.npy',
 'post_sample.csv.gz',
 'post_sample_finetune.csv',
 'products.csv',
 'review_queue.csv.gz',
 'rings.csv',
 'score_percentiles.npy',
 'svm_model.joblib']

## Summary

<div dir="rtl">

- **ممنوعیت ۲۰۱۶:** ۱۱٬۹۱۷ نظر قبل از ممنوعیت جمله‌ی اعلام دارند و بعد از آن فقط ۴۸ نظر. سهم این نظرها در ژوئن ۲۰۱۶ به ۷.۲٪ رسید و بعد از ممنوعیت صفر شد.
- **تورم امتیاز:** داخل یک محصول، نظرهای اعلام‌شده ۰.۵۱ ستاره بالاتر از بقیه‌اند. محصول‌هایی که این نظرها را داشتند بعد از ممنوعیت ۰.۱۲ ستاره بیشتر از بقیه افت کردند (difference-in-differences).
- **برچسب بدون برچسب‌زنی دستی:** مثبت‌ها از جمله‌ی اعلام آمدند و ۴٬۲۸۲ نظر با شکل‌های دیگر اعلام از آموزش کنار رفتند. جمله‌های نشانه‌دار از همه‌ی متن‌ها حذف شدند و نشت با وزن کلمه‌های مدل بررسی شد.
- **مدل:** linear SVM روی TF-IDF با AP برابر ۰.۶۱۶ و AUC برابر ۰.۸۶۲ در cross-validation گروه‌بندی‌شده بر اساس نظردهنده. AP یک مدل تصادفی ۰.۱۹۸ و AP «فقط طول متن» ۰.۳۷۸ است. MiniLM fine-tune شده در cross-validation کمی بهتر بود (۰.۶۳۶). بعد از ممنوعیت روی held out دو مدل تقریباً برابرند (۰.۷۱۶ و ۰.۷۲۲) و روی حلقه‌ها مدل fine-tune شده بهتر است (۰.۸۰۳ در برابر ۰.۷۶۲). SVM به‌خاطر وزن قابل توضیح کلمه‌ها و اجرا روی CPU مدل کاری ماند.
- **بعد از ممنوعیت:** نظرهای نظردهنده‌هایی که قبلاً اعلام کرده بودند و در آموزش نبودند AUC برابر ۰.۷۱۴ دارند (۰.۷۲۳ داخل نظرهای ۵ ستاره و ۰.۷۳۲ در سطح نظردهنده). در ۱۰۰۰ نظر اول سهم آن‌ها ۲۴ برابر پایه است و سهم verified به ۰.۶۸ می‌رسد، در برابر حدود ۰.۸۹ برای بقیه‌ی نظرها.
- **حلقه‌ها:** ۲۰۵ نظردهنده در ۱۱ حلقه‌اند و بزرگ‌ترین حلقه ۱۵۹ عضو دارد. میانگین امتیازشان ۴.۴۹ است و فقط ۱۴.۷٪ نظرهایشان verified است. مدل متنی، بدون هیچ اطلاعی از این شبکه، به نظرهای اعضایی که هیچ‌وقت اعلام نکرده‌اند AUC برابر ۰.۷۶۱ می‌دهد.
- **خروجی بیزینسی:** ۱٪ نظرهای بعد از ممنوعیت (۴٬۵۹۷ نظر) برای بررسی علامت خوردند. ۶۸ محصول و چند برند، مثل SOLO با ۱۳.۲٪، نظر علامت‌خورده‌ی غیرعادی زیاد دارند. ولی امتیاز کلی محصول‌ها با کنار گذاشتن این نظرها تقریباً عوض نمی‌شود.

**محدودیت‌ها:**
- شرط PU learning کامل برقرار نیست. شاید کسانی که اعلام می‌کردند با کسانی که اعلام نمی‌کردند فرق داشته باشند، و regex همه‌ی شکل‌های اعلام را نمی‌گیرد، پس بی‌برچسب‌ها کمی آلوده‌اند.
- مدل سبک «نظردهنده‌ی حرفه‌ای» را هم پیدا می‌کند و هر نظردهنده‌ی حرفه‌ای پول نگرفته است. خروجی امتیاز مشکوک بودن است، نه حکم، و برای صف بررسی و کم کردن وزن است.
- verified شاخص دوطرفه است: نظرهای Vine مجازند و verified نیستند، و طرحی که پول را بیرون از آمازون پس می‌دهد نظر verified می‌سازد.
- سیگنال بعد از ممنوعیت در کل فهرست متوسط است (AUC حدود ۰.۷۱) و فقط بالای فهرست قوی است. ۱۰۰ نظر اول به چند نظردهنده‌ی پرکار وابسته است.
- فقط ۹ ماه داده‌ی قبل از ممنوعیت هست و کالاهای ۲۰۱۷ و ۲۰۱۸ کمی فرق دارند. چند کلمه‌ی نوع کالا هم در مدل مانده است.
- آستانه‌ی ۵ محصول مشترک برای حلقه‌ها با آزمون آماری انتخاب نشده است.

</div>